In [1]:
import tensorflow as tf
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
from tensorflow import keras


plt.switch_backend('Agg')

# --- Configuration Constants ---
SCALE_FACTOR = 4
BATCH_SIZE = 8
LR_SIZE = 96
HR_SIZE = LR_SIZE * SCALE_FACTOR
TRAIN_SAMPLES = 800
EPOCHS = 30
PIXEL_LOSS_WEIGHT = 0.001

# Set logging level to error to reduce noise
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print(f"TensorFlow Version: {tf.__version__}")
print("-" * 50)


TensorFlow Version: 2.20.0
--------------------------------------------------


In [2]:
# ----------------------------------------------------------------------
# 1. KERAS OPTIMIZER AND CALLBACK CONFIGURATION
# ----------------------------------------------------------------------

# Using Adam optimizer with a piecewise constant learning rate schedule.

optim_esrgan = keras.optimizers.Adam(
    learning_rate=keras.optimizers.schedules.PiecewiseConstantDecay(
        boundaries=[5000],
        values=[2e-4, 1e-4]  # Slightly different LR for Real-ESRGAN
    ),
    beta_1=0.9,
    beta_2=0.99
)

# Checkpoint the best model weights based on validation loss
best_weights_checkpoint_path = "best-esrgan-model.weights.h5"

save_best_cb = keras.callbacks.ModelCheckpoint(
    filepath=best_weights_checkpoint_path,
    monitor="loss",
    save_best_only=True,
    save_weights_only=True,
    save_freq="epoch",
)


In [3]:
# ----------------------------------------------------------------------
# 1.5. QUANTITATIVE METRICS (PSNR and SSIM)
# ----------------------------------------------------------------------

def psnr_metric(y_true, y_pred):
    """Peak Signal-to-Noise Ratio (PSNR) calculated over normalized [0, 1] images."""
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim_metric(y_true, y_pred):
    """Structural Similarity Index Measure (SSIM) calculated over normalized [0, 1] images."""
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))


In [4]:
# ----------------------------------------------------------------------
# 2. DATA AUGMENTATION FUNCTIONS (TensorFlow Operators)
# ----------------------------------------------------------------------

def flip_left_right(lowres_img, highres_img):
    """Flips Images to left and right."""
    rn = tf.random.uniform(shape=(), maxval=1)
    return tf.cond(
        rn < 0.5,
        lambda: (lowres_img, highres_img),
        lambda: (
            tf.image.flip_left_right(lowres_img),
            tf.image.flip_left_right(highres_img),
        ),
    )

def random_rotate(lowres_img, highres_img):
    """Rotates Images by 90 degrees."""
    rn = tf.random.uniform(shape=(), maxval=4, dtype=tf.int32)
    return tf.image.rot90(lowres_img, rn), tf.image.rot90(highres_img, rn)


In [5]:
# ----------------------------------------------------------------------
# 3. DATASET LOADING (Updated with Augmentation)
# ----------------------------------------------------------------------

def load_or_simulate_dataset(num_samples, batch_size, lr_shape, hr_shape):
    """
    Loads the real DIV2K dataset via TFDS and applies preprocessing and augmentation.
    """
    try:
        print("Attempting to load real DIV2K dataset via TFDS...")

        train_ds, info = tfds.load(
            'div2k/bicubic_x4',
            split='train',
            as_supervised=True,
            with_info=True
        )

        def filter_min_size(hr_img, lr_img):
            hr_shape = tf.shape(hr_img)
            hr_h, hr_w = hr_shape[0], hr_shape[1]
            return tf.logical_and(hr_h >= HR_SIZE, hr_w >= HR_SIZE)

        initial_samples = info.splits['train'].num_examples
        train_ds = train_ds.filter(filter_min_size)

        print(f"Initial TFDS samples: {initial_samples}")

        def preprocess_image_pair(hr_img, _lr_img_unused):
            hr_img = tf.image.convert_image_dtype(hr_img, tf.float32)

            hr_shape = tf.shape(hr_img)
            hr_h, hr_w = hr_shape[0], hr_shape[1]

            max_offset_h = hr_h - HR_SIZE
            max_offset_w = hr_w - HR_SIZE

            offset_h = tf.random.uniform(shape=[], minval=0, maxval=max_offset_h + 1, dtype=tf.int32)
            offset_w = tf.random.uniform(shape=[], minval=0, maxval=max_offset_w + 1, dtype=tf.int32)

            offset_h = (offset_h // SCALE_FACTOR) * SCALE_FACTOR
            offset_w = (offset_w // SCALE_FACTOR) * SCALE_FACTOR

            hr_patch = tf.image.crop_to_bounding_box(hr_img, offset_h, offset_w, HR_SIZE, HR_SIZE)

            # Generate synthetic LR from HR patch using second-order degradation
            lr_patch = generate_synthetic_lr(hr_patch, scale=SCALE_FACTOR)

            lr_patch, hr_patch = flip_left_right(lr_patch, hr_patch)
            lr_patch, hr_patch = random_rotate(lr_patch, hr_patch)

            return lr_patch, hr_patch

        # Apply preprocessing, shuffling, and batching
        dataset = train_ds.map(preprocess_image_pair, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.shuffle(buffer_size=10).batch(batch_size).prefetch(tf.data.AUTOTUNE)

        print(f"TFDS DIV2K dataset processing configured with BATCH_SIZE={batch_size}.")
        return dataset

    except Exception as e:
        print(f"TFDS Loading failed: {e}. Falling back to simulation.")

        print(f"Creating SIMULATED DIV2K dataset: {num_samples} samples, batch size {batch_size}")
        lr_data = np.random.rand(num_samples, *lr_shape).astype(np.float32)
        hr_data = np.random.rand(num_samples, *hr_shape).astype(np.float32)
        dataset = tf.data.Dataset.from_tensor_slices((lr_data, hr_data))
        dataset = dataset.shuffle(buffer_size=100).batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return dataset


In [6]:
# ----------------------------------------------------------------------
# 3.5. TEST DATASET (BSD100) - LR generated via bicubic downsampling
# ----------------------------------------------------------------------

def load_test_dataset(scale=SCALE_FACTOR):
    """
    Loads the BSD100 dataset for testing, generating LR images via bicubic downsampling
    and pairing them with the HR ground truth.
    """
    try:
        print("\nAttempting to load BSD100 (B100) Test Set via TFDS...")

        try:
            test_ds = tfds.load('bsd100', split='test', as_supervised=True)
        except:
            # Fallback for datasets without a 'test' split (e.g., Set5/14)
            test_ds = tfds.load('bsd100', split='all', as_supervised=True)

        def preprocess_test_image(hr_img, _):
            hr_img = tf.image.convert_image_dtype(hr_img, tf.float32)

            #Generate LR image using Bicubic downsampling (as per Unified Preprocessing standard)
            hr_h = tf.shape(hr_img)[0]
            hr_w = tf.shape(hr_img)[1]

            # Calculate LR dimensions
            lr_h = hr_h // scale
            lr_w = hr_w // scale

            # Downsample using Bicubic method
            lr_img = tf.image.resize(
                hr_img,
                size=[lr_h, lr_w],
                method=tf.image.ResizeMethod.BICUBIC
            )

            # Return LR input and HR ground truth
            return lr_img, hr_img

        # Apply preprocessing (downsampling and normalization)
        test_dataset = test_ds.map(preprocess_test_image, num_parallel_calls=tf.data.AUTOTUNE)

        # Batch size of 1 is typical for evaluation to handle variable image sizes
        test_dataset = test_dataset.batch(1).prefetch(tf.data.AUTOTUNE)

        print("BSD100 Test Set configured for evaluation (LR generated via Bicubic downsampling).")
        return test_dataset

    except Exception as e:
        print(f"TFDS BSD100 Loading failed: {e}. Returning None.")
        return None


In [7]:
# ----------------------------------------------------------------------
# 4.5. REAL-WORLD DEGRADATION (Second-order, simplified)
# ----------------------------------------------------------------------

# The pipeline follows (approx.) Real-ESRGAN: blur -> resize -> noise -> JPEG, twice.
# This simplified TF version remains differentiable where possible and uses JPEG via encode/decode.

def _random_gaussian_kernel(kernel_size=7, sigma_min=0.2, sigma_max=3.0):
    # Create 2D gaussian kernel with random sigma
    ax = tf.range(-(kernel_size//2), kernel_size//2 + 1, dtype=tf.float32)
    xx, yy = tf.meshgrid(ax, ax)
    sigma = tf.random.uniform([], sigma_min, sigma_max)
    kernel = tf.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel = kernel / tf.reduce_sum(kernel)
    kernel = tf.reshape(kernel, [kernel_size, kernel_size, 1, 1])
    kernel = tf.repeat(kernel, repeats=3, axis=2)
    return kernel

@tf.function
def _apply_blur(img):
    ksz = tf.random.uniform([], minval=5, maxval=9, dtype=tf.int32)
    ksz = ksz + 1 - (ksz % 2)  # ensure odd
    kernel = _random_gaussian_kernel(kernel_size=ksz)
    img = tf.expand_dims(img, 0)
    out = tf.nn.depthwise_conv2d(img, tf.concat([kernel, kernel, kernel], axis=3) if False else kernel, strides=[1,1,1,1], padding='SAME')
    return tf.squeeze(out, 0)

@tf.function
def _apply_random_resize(img):
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]
    # random scale in [0.5, 1.0] down or [1.0, 1.5] up, then back to original
    scale = tf.where(tf.random.uniform([]) < 0.5,
                     tf.random.uniform([], 0.5, 1.0),
                     tf.random.uniform([], 1.0, 1.5))
    nh = tf.cast(tf.maximum(2, tf.cast(tf.round(tf.cast(h, tf.float32) * scale), tf.int32)), tf.int32)
    nw = tf.cast(tf.maximum(2, tf.cast(tf.round(tf.cast(w, tf.float32) * scale), tf.int32)), tf.int32)
    method = tf.random.uniform([], 0, 3, dtype=tf.int32)
    methods = [tf.image.ResizeMethod.BILINEAR, tf.image.ResizeMethod.BICUBIC, tf.image.ResizeMethod.AREA]
    img = tf.image.resize(img, [nh, nw], method=methods[method])
    img = tf.image.resize(img, [h, w], method=methods[method])
    return img

@tf.function
def _apply_noise(img):
    # Additive Gaussian noise
    std = tf.random.uniform([], 0.0, 0.05)
    noise = tf.random.normal(tf.shape(img), mean=0.0, stddev=std)
    return tf.clip_by_value(img + noise, 0.0, 1.0)

@tf.function
def _apply_jpeg(img):
    # JPEG compress/decompress with random quality
    q = tf.random.uniform([], 60, 100)
    img_u8 = tf.cast(tf.round(img * 255.0), tf.uint8)
    enc = tf.io.encode_jpeg(img_u8, format='rgb', quality=tf.cast(q, tf.int32))
    dec = tf.io.decode_jpeg(enc)
    return tf.cast(dec, tf.float32) / 255.0

@tf.function
def degrade_once(hr_img):
    x = _apply_blur(hr_img)
    x = _apply_random_resize(x)
    x = _apply_noise(x)
    x = _apply_jpeg(x)
    return x

@tf.function
def degrade_twice(hr_img):
    x = degrade_once(hr_img)
    x = degrade_once(x)
    return x

@tf.function
def generate_synthetic_lr(hr_img, scale=SCALE_FACTOR):
    # Apply second-order degradation at HR size then downscale to LR
    degraded = degrade_twice(hr_img)
    h = tf.shape(hr_img)[0] // scale
    w = tf.shape(hr_img)[1] // scale
    lr = tf.image.resize(degraded, [h, w], method=tf.image.ResizeMethod.AREA)
    return lr


In [8]:
# ----------------------------------------------------------------------
# 4. MODEL DEFINITION - Real-ESRGAN Architecture
# ----------------------------------------------------------------------

def ResidualDenseBlock(x, num_filters=64, growth_channel=32):
    """Dense Block as used in Real-ESRGAN (Residual in Residual Dense Block)."""
    x1 = tf.keras.layers.Conv2D(filters=growth_channel, kernel_size=3, padding='same')(x)
    x1 = tf.keras.layers.LeakyReLU(alpha=0.2)(x1)
    
    x2_input = tf.keras.layers.Concatenate()([x, x1])
    x2 = tf.keras.layers.Conv2D(filters=growth_channel, kernel_size=3, padding='same')(x2_input)
    x2 = tf.keras.layers.LeakyReLU(alpha=0.2)(x2)
    
    x3_input = tf.keras.layers.Concatenate()([x, x1, x2])
    x3 = tf.keras.layers.Conv2D(filters=growth_channel, kernel_size=3, padding='same')(x3_input)
    x3 = tf.keras.layers.LeakyReLU(alpha=0.2)(x3)
    
    x4_input = tf.keras.layers.Concatenate()([x, x1, x2, x3])
    x4 = tf.keras.layers.Conv2D(filters=growth_channel, kernel_size=3, padding='same')(x4_input)
    x4 = tf.keras.layers.LeakyReLU(alpha=0.2)(x4)
    
    x5_input = tf.keras.layers.Concatenate()([x, x1, x2, x3, x4])
    x5 = tf.keras.layers.Conv2D(filters=num_filters, kernel_size=3, padding='same')(x5_input)
    
    return x5

def ResidualInResidualDenseBlock(x, num_filters=64, growth_channel=32):
    """RRDB (Residual in Residual Dense Block) as used in Real-ESRGAN."""
    x1 = ResidualDenseBlock(x, num_filters, growth_channel)
    out1 = x + x1 * 0.2
    
    x2 = ResidualDenseBlock(out1, num_filters, growth_channel)
    out2 = out1 + x2 * 0.2
    
    x3 = ResidualDenseBlock(out2, num_filters, growth_channel)
    out3 = out2 + x3 * 0.2
    
    return out3

def create_esrgan_sr_model(scale=SCALE_FACTOR, lr_size=LR_SIZE):
    """Defines a Real-ESRGAN based Super-Resolution model structure."""
    print("\nDefining Keras Real-ESRGAN Super-Resolution Model...")

    input_tensor = tf.keras.Input(shape=(None, None, 3))

    # 1. Feature Extraction (Initial Conv)
    x = tf.keras.layers.Conv2D(filters=64, kernel_size=3, padding='same')(input_tensor)
    global_res = x

    # 2. RRDB Blocks (Real-ESRGAN typically uses 23, but we use 16 for efficiency)
    for i in range(16):
        x = ResidualInResidualDenseBlock(x, num_filters=64, growth_channel=32)

    # Global Residual Skip Connection
    x = tf.keras.layers.Conv2D(filters=64, kernel_size=3, padding='same')(x)
    x = tf.keras.layers.Add()([x, global_res])

    # 3. Upscaling using Pixel Shuffle (more efficient than UpSampling2D)
    # For x4 upscaling: do 2x upscaling twice (each requires 4x channels for pixel shuffle)
    if scale == 4:
        # First 2x upscale
        x = tf.keras.layers.Conv2D(filters=64*4, kernel_size=3, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
        x = pixel_shuffle(x, 2)
        
        # Second 2x upscale
        x = tf.keras.layers.Conv2D(filters=64*4, kernel_size=3, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
        x = pixel_shuffle(x, 2)
    elif scale == 2:
        # Single 2x upscale
        x = tf.keras.layers.Conv2D(filters=64*4, kernel_size=3, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
        x = pixel_shuffle(x, 2)
    else:
        # For other scales, fall back to regular upsampling
        x = tf.keras.layers.Conv2D(filters=64, kernel_size=3, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
        x = tf.keras.layers.UpSampling2D(size=(scale, scale), interpolation='bilinear')(x)

    # 4. Reconstruction (Final layer)
    output_tensor = tf.keras.layers.Conv2D(filters=3, kernel_size=3, padding='same')(x)
    output_tensor = tf.nn.tanh(output_tensor) * 0.58 + 0.5  # Scale to [0, 1]
    output_tensor = tf.keras.layers.Lambda(lambda x: tf.clip_by_value(x, 0.0, 1.0))(output_tensor)

    model = tf.keras.Model(inputs=input_tensor, outputs=output_tensor, name="Real_ESRGAN_Model")

    # Compile the model using L1 Loss (similar to EDSR), the custom Adam optimizer, and quantitative metrics
    model.compile(optimizer=optim_esrgan, loss='mae', metrics=[psnr_metric, ssim_metric])
    print("Real-ESRGAN Model defined and compiled successfully using **L1 Loss**, Custom Adam Optimizer, and PSNR/SSIM Metrics.")

    return model

def pixel_shuffle(input_tensor, upscale_factor):
    """
    Pixel Shuffle layer implementation in TensorFlow/Keras.
    This is more efficient than using UpSampling2D.
    """
    def shuffle_func(x):
        batch_size = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = tf.shape(x)[3]
        
        # Ensure output channels is correct
        output_channels = C // (upscale_factor ** 2)
        
        # Reshape to separate the spatial and channel dimensions
        x = tf.reshape(x, [batch_size, H, W, upscale_factor, upscale_factor, output_channels])
        
        # Transpose to place upscale_factor dimensions in the correct position
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        
        # Reshape to merge the upscale_factor dimensions
        x = tf.reshape(x, [batch_size, H * upscale_factor, W * upscale_factor, output_channels])
        
        return x
    
    return tf.keras.layers.Lambda(shuffle_func)(input_tensor)


In [9]:
# ----------------------------------------------------------------------
# 5. EVALUATION COMPONENTS
# ----------------------------------------------------------------------

def predict_super_resolution(model, lr_image):
    """
    Uses the trained Keras model to generate a super-resolved image.
    Accepts normalized [0, 1] tensor.
    """
    # Ensure it's batched
    lr_batched = tf.expand_dims(lr_image, axis=0)

    # Model prediction (outputs float [0, 1])
    sr_float_batched = model.predict(lr_batched, verbose=0)
    sr_float = tf.squeeze(sr_float_batched, axis=0)

    # Convert back to uint8 [0, 255] for display
    sr_image_uint8 = tf.cast(tf.clip_by_value(sr_float * 255.0, 0, 255), tf.uint8)

    return sr_image_uint8, sr_float # Return both uint8 for display and float for metrics

def plot_lr_sr_hr(lr_image, sr_image, hr_image, psnr, ssim, index, scale=SCALE_FACTOR):
    """
    Displays the LR Input, SR Output, and HR Ground Truth comparison by saving the plot to a file.
    Includes calculated metrics for the current sample.
    """
    # Determine the display size
    HR_SIZE_LOCAL = hr_image.shape[0]

    # Create the figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Resize LR for visualization only (using nearest neighbor for clarity)
    lr_display = tf.image.resize(tf.cast(lr_image * 255.0, tf.uint8).numpy(), [HR_SIZE_LOCAL, HR_SIZE_LOCAL], method='nearest').numpy().astype(np.uint8)

    # Convert HR and SR (uint8) for display
    hr_display = tf.cast(hr_image * 255.0, tf.uint8).numpy()
    sr_display = sr_image.numpy()

    axes[0].imshow(lr_display)
    axes[0].set_title(f"Low Resolution Input (x{scale} Bicubic)", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(sr_display.astype(np.uint8))
    axes[1].set_title(f"Real-ESRGAN Output (PSNR: {psnr:.2f} dB, SSIM: {ssim:.4f})", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(hr_display.astype(np.uint8))
    axes[2].set_title(f"High Resolution Ground Truth", fontsize=10)
    axes[2].axis("off")

    plt.suptitle(f"BSD100 Test Sample {index+1}", fontsize=12)
    plt.tight_layout()

    # Save the plot to a file instead of trying to show it interactively
    filepath = f"esrgan_bsd100_test_comparison_sample_{index+1}.png"
    plt.savefig(filepath)
    plt.close(fig) # Close the figure to free up memory
    print(f"Plot saved to {filepath}")

def run_test_evaluation_bsd100(model, test_dataset, num_samples=5):
    """
    Runs evaluation on the BSD100 dataset, calculating metrics and saving plots.
    """
    print(f"\n--- Starting BSD100 Test Evaluation with Real-ESRGAN ({num_samples} Samples) ---")

    total_psnr = 0.0
    total_ssim = 0.0
    count = 0

    # Iterate over the first few samples for visual plotting
    for i, (lr_batch, hr_batch) in enumerate(test_dataset.take(num_samples)):
        if i >= num_samples:
            break

        # Extract the single image from the batch
        lr_img_norm = lr_batch[0] # Normalized LR [0, 1]
        hr_img_norm = hr_batch[0] # Normalized HR [0, 1]

        # Upscale the image
        sr_img_uint8, sr_img_norm = predict_super_resolution(model, lr_img_norm)

        # Calculate metrics for this sample
        current_psnr = psnr_metric(hr_img_norm, sr_img_norm).numpy()
        # tf.image.ssim returns a single value if both inputs have the same shape
        current_ssim = tf.image.ssim(hr_img_norm, sr_img_norm, max_val=1.0).numpy()

        # Accumulate metrics
        total_psnr += current_psnr
        total_ssim += current_ssim
        count += 1

        # Plot the LR, SR, and HR results, including metrics
        plot_lr_sr_hr(lr_img_norm, sr_img_uint8, hr_img_norm, current_psnr, current_ssim, i)

    # Calculate and print final mean metrics if data was processed
    if count > 0:
        mean_psnr = total_psnr / count
        mean_ssim = total_ssim / count
        print(f"\n--- RESULTS ON BSD100 TEST SET (First {count} Samples) ---")
        print(f"Mean PSNR: {mean_psnr:.4f} dB")
        print(f"Mean SSIM: {mean_ssim:.4f}")
    else:
        print("No BSD100 test samples were available for evaluation.")

    print("--- BSD100 Test Evaluation Complete. ---")


In [10]:
# ----------------------------------------------------------------------
# 7. DISCRIMINATOR (U-Net style, logits output for RaGAN)
# ----------------------------------------------------------------------

def conv_block(x, filters, k=3, s=1):
    x = tf.keras.layers.Conv2D(filters, k, strides=s, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    return x


def down_block(x, filters):
    x = conv_block(x, filters)
    x = conv_block(x, filters)
    skip = x
    x = tf.keras.layers.Conv2D(filters, 4, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    return x, skip


def up_block(x, skip, filters):
    x = tf.keras.layers.UpSampling2D(size=2, interpolation='bilinear')(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    x = conv_block(x, filters)
    return x


def build_discriminator(input_shape=(None, None, 3)):
    inputs = tf.keras.Input(shape=input_shape)

    x1, s1 = down_block(inputs, 64)   # 1/2
    x2, s2 = down_block(x1, 128)      # 1/4
    x3, s3 = down_block(x2, 256)      # 1/8
    x4, s4 = down_block(x3, 512)      # 1/16

    b = conv_block(x4, 512)
    b = conv_block(b, 512)

    u3 = up_block(b, s4, 256)
    u2 = up_block(u3, s3, 128)
    u1 = up_block(u2, s2, 64)
    u0 = up_block(u1, s1, 64)

    logits_map = tf.keras.layers.Conv2D(1, 1, padding='same')(u0)
    pooled = tf.keras.layers.GlobalAveragePooling2D()(logits_map)  # scalar logit

    model = tf.keras.Model(inputs, pooled, name='UNetDiscriminator')
    return model


In [11]:
# ----------------------------------------------------------------------
# 8. PERCEPTUAL LOSS (VGG19 feature extractor)
# ----------------------------------------------------------------------

def build_vgg_feature_extractor(layer_name='block5_conv4'):
    # VGG expects inputs in range [0, 255] after preprocessing
    vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
    vgg.trainable = False
    outputs = vgg.get_layer(layer_name).output
    model = tf.keras.Model(vgg.input, outputs, name='VGG19_Feature_Extractor')
    model.trainable = False
    return model

@tf.function
def perceptual_loss(vgg, y_true, y_pred):
    # Convert [0,1] to VGG expected range using preprocess_input
    def to_vgg(x):
        x255 = x * 255.0
        return tf.keras.applications.vgg19.preprocess_input(x255)
    f_true = vgg(to_vgg(y_true))
    f_pred = vgg(to_vgg(y_pred))
    return tf.reduce_mean(tf.abs(f_true - f_pred))


In [12]:
# ----------------------------------------------------------------------
# 9. GAN TRAINING MODEL (custom train_step for G and D)
# ----------------------------------------------------------------------

class RealESRGANTrainer(tf.keras.Model):
    def __init__(self, generator, discriminator, vgg_feature_extractor,
                 lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005,
                 **kwargs):
        super().__init__(**kwargs)
        self.generator = generator
        self.discriminator = discriminator
        self.vgg = vgg_feature_extractor
        self.lambda_pixel = lambda_pixel
        self.lambda_percep = lambda_percep
        self.lambda_adv = lambda_adv

    def compile(self, g_optimizer, d_optimizer):
        super().compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_tracker_g = tf.keras.metrics.Mean(name="g_loss")
        self.loss_tracker_d = tf.keras.metrics.Mean(name="d_loss")
        self.psnr_metric_tracker = tf.keras.metrics.Mean(name="psnr")
        self.ssim_metric_tracker = tf.keras.metrics.Mean(name="ssim")

    @property
    def metrics(self):
        return [self.loss_tracker_g, self.loss_tracker_d, self.psnr_metric_tracker, self.ssim_metric_tracker]

    def compute_generator_losses(self, lr, hr, sr):
        # Pixel (L1) loss
        pixel = tf.reduce_mean(tf.abs(hr - sr))
        # Perceptual loss
        percep = perceptual_loss(self.vgg, hr, sr)
        # RaGAN generator loss: real-vs-fake with relativistic average
        d_real = self.discriminator(hr, training=True)
        d_fake = self.discriminator(sr, training=True)
        mean_real = tf.reduce_mean(d_real)
        mean_fake = tf.reduce_mean(d_fake)
        # Generator wants D_fake - E[D_real] to be classified as real
        g_adv = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=tf.ones_like(d_fake), logits=d_fake - mean_real)
        adv = tf.reduce_mean(g_adv)
        total = self.lambda_pixel * pixel + self.lambda_percep * percep + self.lambda_adv * adv
        return total, pixel, percep, adv

    def compute_discriminator_loss(self, hr, sr):
        d_real = self.discriminator(hr, training=True)
        d_fake = self.discriminator(sr, training=True)
        mean_real = tf.reduce_mean(d_real)
        mean_fake = tf.reduce_mean(d_fake)
        # Discriminator: classify D_real - E[D_fake] as real, D_fake - E[D_real] as fake
        real_loss = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=tf.ones_like(d_real), logits=d_real - mean_fake)
        fake_loss = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=tf.zeros_like(d_fake), logits=d_fake - mean_real)
        return tf.reduce_mean(real_loss) + tf.reduce_mean(fake_loss)

    def train_step(self, data):
        lr, hr = data

        # 1) Update Discriminator
        with tf.GradientTape() as d_tape:
            sr = self.generator(lr, training=True)
            d_loss = self.compute_discriminator_loss(hr, tf.stop_gradient(sr))
        d_grads = d_tape.gradient(d_loss, self.discriminator.trainable_variables)
        self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))

        # 2) Update Generator
        with tf.GradientTape() as g_tape:
            sr = self.generator(lr, training=True)
            g_total, l_pix, l_perc, l_adv = self.compute_generator_losses(lr, hr, sr)
        g_grads = g_tape.gradient(g_total, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_variables))

        # Metrics
        self.loss_tracker_g.update_state(g_total)
        self.loss_tracker_d.update_state(d_loss)
        self.psnr_metric_tracker.update_state(tf.reduce_mean(tf.image.psnr(hr, sr, max_val=1.0)))
        self.ssim_metric_tracker.update_state(tf.reduce_mean(tf.image.ssim(hr, sr, max_val=1.0)))

        return {m.name: m.result() for m in self.metrics}


In [13]:
# ----------------------------------------------------------------------
# 10. TRAINING WIRING: use GAN trainer instead of plain generator
# ----------------------------------------------------------------------

# Helper to build the full GAN training object

def build_gan_trainer(scale=SCALE_FACTOR):
    generator = create_esrgan_sr_model(scale=scale)
    discriminator = build_discriminator(input_shape=(None, None, 3))
    vgg = build_vgg_feature_extractor('block5_conv4')

    trainer = RealESRGANTrainer(generator, discriminator, vgg,
                                lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005)

    # Separate optimizers for G and D
    g_opt = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.9, beta_2=0.99)
    d_opt = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.9, beta_2=0.99)

    trainer.compile(g_optimizer=g_opt, d_optimizer=d_opt)
    return trainer


In [15]:
# ----------------------------------------------------------------------
# 11. EXECUTION AND EVALUATION (GAN)
# ----------------------------------------------------------------------

if __name__ == '__main__':
    # 1. Initialize the training dataset (DIV2K)
    train_dataset = load_or_simulate_dataset(
        num_samples=TRAIN_SAMPLES,
        batch_size=BATCH_SIZE,
        lr_shape=(LR_SIZE, LR_SIZE, 3),
        hr_shape=(HR_SIZE, HR_SIZE, 3)
    )

    # 1.5. Initialize the test dataset (BSD100)
    test_dataset = load_test_dataset()

    # 2. Build GAN trainer (Generator + Discriminator + VGG perceptual)
    gan_trainer = build_gan_trainer(scale=SCALE_FACTOR)

    print("\n" * 2)

    # 3. Training (GAN)
    print(f"--- Starting Real-ESRGAN GAN Training ({EPOCHS} Epochs) ---")
    gan_trainer.fit(
        train_dataset,
        epochs=EPOCHS,
        verbose=1
    )
    print("--- Training Complete. Starting Evaluation. ---")

    try:
        # 4. Evaluation using BSD100 (use underlying generator for inference)
        generator = gan_trainer.generator
        if test_dataset:
            run_test_evaluation_bsd100(generator, test_dataset, num_samples=5)
        else:
            print("\nWARNING: Could not load BSD100. Running ad-hoc visual check on training data.")
            def run_ad_hoc_evaluation_fallback(model, dataset, num_samples=8):
                print(f"\n--- Starting Ad-Hoc Visual Evaluation ({num_samples} Samples, Model: Real-ESRGAN) ---")
                for i, (lr_batch, _) in enumerate(dataset.take(num_samples)):
                    if i >= num_samples: break
                    lowres_img = lr_batch[0]
                    sr_img_uint8, _ = predict_super_resolution(model, lowres_img)
                    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                    lr_display = tf.image.resize(tf.cast(lowres_img * 255.0, tf.uint8).numpy(), [sr_img_uint8.shape[0], sr_img_uint8.shape[1]], method='nearest').numpy().astype(np.uint8)
                    axes[0].imshow(lr_display); axes[0].axis("off"); axes[0].set_title("Low Resolution Input", fontsize=10)
                    axes[1].imshow(sr_img_uint8.numpy().astype(np.uint8)); axes[1].axis("off"); axes[1].set_title("Real-ESRGAN Super-Resolution Output", fontsize=10)
                    plt.tight_layout(); filepath = f"esrgan_sr_comparison_sample_{i+1}_fallback.png"; plt.savefig(filepath); plt.close(fig)
                    print(f"Fallback plot saved to {filepath}")
                print("--- Ad-Hoc Visual Evaluation Complete. ---")
            run_ad_hoc_evaluation_fallback(generator, train_dataset, num_samples=8)

        print("\nReal-ESRGAN Model Evaluation successfully completed. ")

    except Exception as e:
        print(f"\nFATAL ERROR: The script failed unexpectedly during evaluation.")
        print(f"Error detail: {e}")
        sys.exit(1)



Attempting to load real DIV2K dataset via TFDS...
Initial TFDS samples: 800
TFDS Loading failed: in user code:

    File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_29852\2931029555.py", line 47, in preprocess_image_pair  *
        lr_patch = generate_synthetic_lr(hr_patch, scale=SCALE_FACTOR)
    File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_29852\1114607729.py", line 77, in generate_synthetic_lr  *
        degraded = degrade_twice(hr_img)
    File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_29852\1114607729.py", line 70, in degrade_twice  *
        x = degrade_once(hr_img)
    File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_29852\1114607729.py", line 63, in degrade_once  *
        x = _apply_random_resize(x)
    File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_29852\1114607729.py", line 40, in _apply_random_resize  *
        img = tf.image.resize(img, [nh, nw], method=methods[method])

    TypeError: list indices must be integers or sl

C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```
